# Data Types and Formatting

Incorrect data types and inconsistent formatting are among the most common yet overlooked data quality issues. A numeric column loaded as `object` silently blocks all arithmetic; a date stored as a string prevents any temporal operation; trailing whitespace in a category field creates phantom duplicate categories.

### What you will learn

| Section | Topic |
|---------|-------|
| 1 | Inspecting and auditing column types |
| 2 | Coercing mixed-type numeric columns |
| 3 | Parsing and normalising date/time columns |
| 4 | String normalisation — whitespace, case, regex |
| 5 | Categorical dtype — memory and performance benefits |
| 6 | Memory optimisation with narrow types |

In [1]:
import numpy as np
import pandas as pd
import re
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)

print('Setup complete.')

Setup complete.


In [3]:
# --- Build a deliberately messy DataFrame that mirrors real-world ingestion issues ---
raw_data = {
    'customer_id': ['C001', 'C002', 'C003', 'C004', 'C005', 'C006', 'C007'],
    # Numeric column with string sentinels and formatting noise
    'revenue':     ['1,250.50', '840.00', 'N/A', '-', '3,100', '920.75', 'n/a'],
    # Age with a stray string entry
    'age':         ['34', '28', '45', 'unknown', '52', '39', '31'],
    # Dates in multiple formats — common after merging sources
    'signup_date': ['2023-01-15', '15/02/2023', 'March 3, 2023',
                    '2023-04-22', '05-05-2023', '2023-06-30', '7/7/2023'],
    # Categorical column with whitespace, mixed case, and typos
    'plan':        ['  premium', 'Basic', 'PREMIUM', 'basic ', 'Basicc', 'Premium', '  basic'],
    # Boolean stored as strings
    'is_active':   ['True', 'False', 'true', '1', '0', 'YES', 'no'],
    # Country codes with inconsistent formatting
    'country':     ['us', 'GB', 'de ', ' FR', 'US', 'gb', 'De'],
}

df = pd.DataFrame(raw_data)
print('Raw DataFrame:')
print(df.to_string())
print('\nDtypes:')
print(df.dtypes)

Raw DataFrame:
  customer_id   revenue      age    signup_date       plan is_active country
0        C001  1,250.50       34     2023-01-15    premium      True      us
1        C002    840.00       28     15/02/2023      Basic     False      GB
2        C003       N/A       45  March 3, 2023    PREMIUM      true     de 
3        C004         -  unknown     2023-04-22     basic          1      FR
4        C005     3,100       52     05-05-2023     Basicc         0      US
5        C006    920.75       39     2023-06-30    Premium       YES      gb
6        C007       n/a       31       7/7/2023      basic        no      De

Dtypes:
customer_id    object
revenue        object
age            object
signup_date    object
plan           object
is_active      object
country        object
dtype: object


---
## Section 1 — Inspecting and Auditing Column Types

Before fixing anything, build a complete picture of what each column actually contains.

In [4]:
def audit_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Return a per-column audit summary."""
    rows = []
    for col in df.columns:
        s = df[col]
        rows.append({
            'column'         : col,
            'dtype'          : str(s.dtype),
            'n_missing'      : s.isnull().sum(),
            'pct_missing'    : f"{s.isnull().mean()*100:.1f}%",
            'n_unique'       : s.nunique(),
            'sample_values'  : s.dropna().unique()[:3].tolist(),
        })
    return pd.DataFrame(rows).set_index('column')

audit_dataframe(df)

,dtype,n_missing,pct_missing,n_unique,sample_values
column,,,,,
customer_id,object,0,0.0%,7,"[C001, C002, C003]"
revenue,object,0,0.0%,7,"[1,250.50, 840.00, N/A]"
age,object,0,0.0%,7,"[34, 28, 45]"
signup_date,object,0,0.0%,7,"[2023-01-15, 15/02/2023, March 3, 2023]"
plan,object,0,0.0%,7,"[ premium, Basic, PREMIUM]"
is_active,object,0,0.0%,7,"[True, False, true]"
country,object,0,0.0%,7,"[us, GB, de ]"


---
## Section 2 — Coercing Mixed-Type Numeric Columns

Numeric columns often arrive as strings due to:
- Thousands separators (`1,250.50`)
- Custom null sentinels (`N/A`, `-`, `null`)
- Units appended as text (`$1250`, `1250 USD`)
- Stray text entries (`unknown`)

`pd.to_numeric(errors='coerce')` is the safest tool — it converts what it can and sets the rest to `NaN`, which you can then handle explicitly.

In [5]:
# Step 1: remove thousands separators and known sentinel strings
def clean_numeric_string(series: pd.Series, sentinels: list = None) -> pd.Series:
    sentinels = sentinels or ['N/A', 'n/a', '-', 'null', 'NULL', 'unknown', 'N.A.']
    s = series.astype(str).str.strip()
    # Replace known sentinels with NaN placeholder before numeric conversion
    s = s.replace(sentinels, np.nan)
    # Remove thousands separators (commas)
    s = s.str.replace(',', '', regex=False)
    return pd.to_numeric(s, errors='coerce')

df['revenue_clean'] = clean_numeric_string(df['revenue'])
df['age_clean']     = pd.to_numeric(df['age'].replace('unknown', np.nan), errors='coerce')

print('Revenue — before vs after:')
for orig, clean in zip(df['revenue'], df['revenue_clean']):
    print(f'  {str(orig):12s}  →  {clean}')

print(f"\nMissing revenue after cleaning: {df['revenue_clean'].isnull().sum()}")
print(f"Missing age after cleaning    : {df['age_clean'].isnull().sum()}")

Revenue — before vs after:
  1,250.50      →  1250.5
  840.00        →  840.0
  N/A           →  nan
  -             →  nan
  3,100         →  3100.0
  920.75        →  920.75
  n/a           →  nan

Missing revenue after cleaning: 3
Missing age after cleaning    : 1


---
## Section 3 — Parsing and Normalising Date/Time Columns

Date columns sourced from multiple systems often contain mixed formats. `pd.to_datetime` with `dayfirst` and `format` options handles most cases; `dateutil.parser` handles ambiguous formats.

**Key pitfalls:**
- `01/02/2023` — is that Jan 2 or Feb 1? Depends on locale.
- Timezones: always store in UTC; localise for display only.
- Excel serial dates: Excel stores dates as integer day counts from 1900-01-01.

In [6]:
# Mixed date formats in one column — the real-world nightmare
print('Raw signup_date values:')
for val in df['signup_date']:
    print(f'  {val}')

# pd.to_datetime with format='mixed' (pandas >= 2.0) infers per-entry format
df['signup_parsed'] = pd.to_datetime(df['signup_date'], format='mixed', dayfirst=False)

print('\nParsed dates:')
for raw, parsed in zip(df['signup_date'], df['signup_parsed']):
    print(f'  {str(raw):20s}  →  {parsed.date()}')

Raw signup_date values:
  2023-01-15
  15/02/2023
  March 3, 2023
  2023-04-22
  05-05-2023
  2023-06-30
  7/7/2023

Parsed dates:
  2023-01-15            →  2023-01-15
  15/02/2023            →  2023-02-15
  March 3, 2023         →  2023-03-03
  2023-04-22            →  2023-04-22
  05-05-2023            →  2023-05-05
  2023-06-30            →  2023-06-30
  7/7/2023              →  2023-07-07


In [7]:
# Extracting useful temporal features from a parsed datetime
df['signup_year']    = df['signup_parsed'].dt.year
df['signup_month']   = df['signup_parsed'].dt.month
df['signup_quarter'] = df['signup_parsed'].dt.quarter
df['signup_weekday'] = df['signup_parsed'].dt.day_name()
df['days_since_signup'] = (pd.Timestamp('2024-01-01') - df['signup_parsed']).dt.days

print(df[['signup_date', 'signup_parsed', 'signup_month', 'signup_quarter',
          'signup_weekday', 'days_since_signup']].to_string())

     signup_date signup_parsed  signup_month  signup_quarter signup_weekday  days_since_signup
0     2023-01-15    2023-01-15             1               1         Sunday                351
1     15/02/2023    2023-02-15             2               1      Wednesday                320
2  March 3, 2023    2023-03-03             3               1         Friday                304
3     2023-04-22    2023-04-22             4               2       Saturday                254
4     05-05-2023    2023-05-05             5               2         Friday                241
5     2023-06-30    2023-06-30             6               2         Friday                185
6       7/7/2023    2023-07-07             7               3         Friday                178


In [8]:
# --- Timezone handling ---
# Always localise to UTC at ingestion; convert to local timezone for display only
# zoneinfo is in the standard library since Python 3.9 — no extra install needed
from zoneinfo import ZoneInfo

dates_with_tz = pd.to_datetime(['2024-01-15 10:30:00', '2024-03-20 14:00:00'])
dates_utc     = dates_with_tz.tz_localize('UTC')
dates_lisbon  = dates_utc.tz_convert('Europe/Lisbon')

print('UTC    :', dates_utc.tolist())
print('Lisbon :', dates_lisbon.tolist())

UTC    : [Timestamp('2024-01-15 10:30:00+0000', tz='UTC'), Timestamp('2024-03-20 14:00:00+0000', tz='UTC')]
Lisbon : [Timestamp('2024-01-15 10:30:00+0000', tz='Europe/Lisbon'), Timestamp('2024-03-20 14:00:00+0000', tz='Europe/Lisbon')]


---
## Section 4 — String Normalisation

String inconsistencies silently multiply category cardinality and cause silent merge failures:

- `'Premium'`, `'PREMIUM'`, `'premium'`, `'  premium'` — four representations of one value
- `'US'`, `'us'`, `'U.S.'` — three representations of one country

A standardised normalisation pipeline should always run before any categorical analysis or encoding.

In [9]:
# --- Plan column: whitespace + case + typo correction ---
print('Unique plan values before normalisation:')
print(df['plan'].unique())

# Standard normalisation pipeline
df['plan_clean'] = (
    df['plan']
      .str.strip()            # remove leading/trailing whitespace
      .str.lower()            # lowercase for consistent comparison
      .str.replace(r'\s+', ' ', regex=True)   # collapse internal spaces
)

# Map known typos to correct values
typo_map = {
    'basicc': 'basic',
    'permium': 'premium',
    'premiun': 'premium',
}
df['plan_clean'] = df['plan_clean'].replace(typo_map)

print('\nUnique plan values after normalisation:')
print(df['plan_clean'].unique())

Unique plan values before normalisation:
['  premium' 'Basic' 'PREMIUM' 'basic ' 'Basicc' 'Premium' '  basic']

Unique plan values after normalisation:
['premium' 'basic']


In [10]:
# --- Country codes: strip + upper + standardise ---
print('Country before:', df['country'].unique())

df['country_clean'] = df['country'].str.strip().str.upper()

print('Country after: ', df['country_clean'].unique())

Country before: ['us' 'GB' 'de ' ' FR' 'US' 'gb' 'De']
Country after:  ['US' 'GB' 'DE' 'FR']


In [11]:
# --- Boolean strings → actual bool ---
TRUTHY = {'true', '1', 'yes', 'y', 'on'}
FALSY  = {'false', '0', 'no', 'n', 'off'}

def parse_bool_string(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip().str.lower()
    result = pd.Series(np.nan, index=series.index, dtype=object)
    result[s.isin(TRUTHY)] = True
    result[s.isin(FALSY)]  = False
    return result.astype('boolean')  # pandas nullable boolean

df['is_active_clean'] = parse_bool_string(df['is_active'])
print('is_active before → after:')
for before, after in zip(df['is_active'], df['is_active_clean']):
    print(f'  {str(before):8s}  →  {after}')

is_active before → after:
  True      →  True
  False     →  False
  true      →  True
  1         →  True
  0         →  False
  YES       →  True
  no        →  False


In [12]:
# --- Regex: extract structured information from free-text strings ---
# Example: product codes embedded in a description field
product_strings = pd.Series([
    'Order #ORD-20451 for product SKU-AX1029',
    'Ref: ORD-19834, item SKU-BX0034',
    'Invoice ORD-21100 — SKU-CX9901 qty 3',
    'ORD-18000 (replacement for SKU-DX2200)',
])

# Extract order and SKU codes
order_id = product_strings.str.extract(r'(ORD-\d+)', expand=False)
sku_code  = product_strings.str.extract(r'(SKU-[A-Z]{2}\d+)', expand=False)

result = pd.DataFrame({'raw': product_strings, 'order_id': order_id, 'sku': sku_code})
print(result.to_string())

                                       raw   order_id         sku
0  Order #ORD-20451 for product SKU-AX1029  ORD-20451  SKU-AX1029
1          Ref: ORD-19834, item SKU-BX0034  ORD-19834  SKU-BX0034
2     Invoice ORD-21100 — SKU-CX9901 qty 3  ORD-21100  SKU-CX9901
3   ORD-18000 (replacement for SKU-DX2200)  ORD-18000  SKU-DX2200


---
## Section 5 — Categorical Dtype

Converting string columns to `pd.Categorical` (or `category` dtype) provides:
- **Memory reduction**: stores category codes as integers; strings stored once
- **Faster groupby / value_counts**: integer comparisons instead of string comparisons
- **Ordered categories**: enables meaningful sorting and comparison (e.g. `small < medium < large`)

Use it on any column with low cardinality relative to row count (rule of thumb: < 50% unique values).

In [13]:
# Generate a larger dataset to demonstrate memory savings
rng = np.random.default_rng(42)
n = 100_000

df_large = pd.DataFrame({
    'product'  : rng.choice(['Widget A', 'Widget B', 'Gadget X', 'Gadget Y', 'Service Z'], n),
    'region'   : rng.choice(['North', 'South', 'East', 'West'], n),
    'tier'     : rng.choice(['bronze', 'silver', 'gold', 'platinum'], n),
    'quantity' : rng.integers(1, 100, n),
})

string_mem = df_large[['product', 'region', 'tier']].memory_usage(deep=True).sum() / 1e6
print(f'Memory as object dtype  : {string_mem:.2f} MB')

# Convert to category
for col in ['product', 'region', 'tier']:
    df_large[col] = df_large[col].astype('category')

cat_mem = df_large[['product', 'region', 'tier']].memory_usage(deep=True).sum() / 1e6
print(f'Memory as category dtype: {cat_mem:.2f} MB')
print(f'Reduction               : {(1 - cat_mem/string_mem)*100:.1f}%')

Memory as object dtype  : 16.57 MB
Memory as category dtype: 0.30 MB
Reduction               : 98.2%


In [13]:
# Ordered categorical — enables meaningful comparison and sorting
tier_order = pd.CategoricalDtype(
    categories=['bronze', 'silver', 'gold', 'platinum'],
    ordered=True
)
df_large['tier'] = df_large['tier'].astype(tier_order)

print('Is gold > silver?', df_large['tier'].cat.categories)

# Demonstrate ordered filtering
high_tier = df_large[df_large['tier'] >= 'gold']
print(f"\nRows with tier >= 'gold': {len(high_tier):,}")
print(high_tier['tier'].value_counts())

Is gold > silver? Index(['bronze', 'silver', 'gold', 'platinum'], dtype='str')



Rows with tier >= 'gold': 49,778
tier
gold        24919
platinum    24859
bronze          0
silver          0
Name: count, dtype: int64


---
## Section 6 — Memory Optimisation with Narrow Types

Pandas defaults to 64-bit types for all numerics. For large datasets, downcasting to the smallest type that fits the data can halve or quarter memory usage with no loss of information.

In [14]:
def downcast_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Downcast numeric columns to the smallest fitting type."""
    df = df.copy()
    for col in df.select_dtypes('integer').columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    for col in df.select_dtypes('float').columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    return df

# Demo on the large synthetic dataset
df_num = df_large[['quantity']].copy()
df_num['price']  = np.random.uniform(1.0, 500.0, len(df_num)).astype('float64')
df_num['rating'] = np.random.randint(1, 6, len(df_num)).astype('int64')

print('Before downcast:')
print(df_num.dtypes)
print(f'Memory: {df_num.memory_usage(deep=True).sum() / 1e6:.3f} MB')

df_down = downcast_dataframe(df_num)
print('\nAfter downcast:')
print(df_down.dtypes)
print(f'Memory: {df_down.memory_usage(deep=True).sum() / 1e6:.3f} MB')

Before downcast:
quantity      int64
price       float64
rating        int64
dtype: object
Memory: 2.400 MB

After downcast:
quantity       int8
price       float32
rating         int8
dtype: object
Memory: 0.600 MB


---
## Cleaned DataFrame Summary

In [15]:
# Assemble the cleaned version of the original messy DataFrame
df_clean = pd.DataFrame({
    'customer_id'       : df['customer_id'],
    'revenue'           : df['revenue_clean'],
    'age'               : df['age_clean'],
    'signup_date'       : df['signup_parsed'],
    'days_since_signup' : df['days_since_signup'],
    'plan'              : df['plan_clean'].astype('category'),
    'country'           : df['country_clean'].astype('category'),
    'is_active'         : df['is_active_clean'],
})

print('Cleaned DataFrame:')
print(df_clean.to_string())
print('\nDtypes after cleaning:')
print(df_clean.dtypes)

Cleaned DataFrame:
  customer_id  revenue   age signup_date  days_since_signup     plan country  is_active
0        C001  1250.50  34.0  2023-01-15                351  premium      US       True
1        C002   840.00  28.0  2023-02-15                320    basic      GB      False
2        C003      NaN  45.0  2023-03-03                304  premium      DE       True
3        C004      NaN   NaN  2023-04-22                254    basic      FR       True
4        C005  3100.00  52.0  2023-05-05                241    basic      US      False
5        C006   920.75  39.0  2023-06-30                185  premium      GB       True
6        C007      NaN  31.0  2023-07-07                178    basic      DE      False

Dtypes after cleaning:
customer_id                  object
revenue                     float64
age                         float64
signup_date          datetime64[ns]
days_since_signup             int64
plan                       category
country                    category
i

---
## Key Takeaways

1. **Audit types immediately after loading** — never assume a numeric column is numeric.
2. **`pd.to_numeric(errors='coerce')` is your safest tool** for mixed-type columns — it preserves NaN for unparseable values instead of raising.
3. **Parse dates at load time** — every downstream temporal operation (lag features, seasonality, duration) depends on a true `datetime64` dtype.
4. **String normalisation must always happen before groupby or value_counts** — invisible whitespace and case differences create phantom duplicate categories.
5. **`category` dtype pays off at > 10k rows** — memory savings of 50–90% on low-cardinality columns are common.
6. **Ordered categoricals** unlock `<`/`>` comparisons and correct sorting — use them for any column with a natural rank (size, tier, severity).

---
## Exercises

1. Load the `seaborn` `mpg` dataset. Identify and fix all type issues (hint: check `horsepower`).
2. The `mpg` dataset has a `model_year` column stored as integers (e.g. 70 = 1970). Convert it to a proper `datetime` (use January 1 of that year).
3. The `name` column contains strings like `'chevrolet chevelle malibu'`. Extract the manufacturer (first word) into a new column and convert it to `category` dtype. How much memory does this save?
4. Write a function `normalize_string_column(series)` that applies strip, lower, and collapses internal spaces, then apply it to the `name` column.

In [17]:
# ========== 1. Import libraries ==========
import pandas as pd
import numpy as np

# ========== 2. Load dataset ==========
file_path = r'C:\Users\35111\Downloads\OneDrive_1_2026-5-26\mpg.csv'
mpg = pd.read_csv(file_path)
print("MPG dataset loaded. Shape:", mpg.shape)
print("Original data types:")
print(mpg.dtypes)
print("\nFirst 5 rows:")
print(mpg.head())

# ========== Exercise 1: Identify and fix type issues (horsepower) ==========
print("\n" + "="*60)
print("Exercise 1: Identify and fix type issues (horsepower)")
print("="*60)

# Check unique values in 'horsepower'
print("Unique values in 'horsepower' (first 20):")
print(mpg['horsepower'].unique()[:20])
# '?' indicates missing values, causing object dtype. Convert to numeric.
mpg['horsepower'] = pd.to_numeric(mpg['horsepower'], errors='coerce')
print("After conversion, horsepower dtype:", mpg['horsepower'].dtype)
print("Number of missing values in horsepower:", mpg['horsepower'].isnull().sum())

# Check other columns
print("\nAll dtypes after fixing horsepower:")
print(mpg.dtypes)

# ========== Exercise 2: Convert model_year to datetime (Jan 1 of each year) ==========
print("\n" + "="*60)
print("Exercise 2: Convert model_year to datetime (Jan 1 of each year)")
print("="*60)

# model_year is two‑digit (70 → 1970)
mpg['year'] = 1900 + mpg['model_year']
mpg['model_date'] = pd.to_datetime(mpg['year'], format='%Y')
print("Sample of converted dates:")
print(mpg[['model_year', 'year', 'model_date']].head(10))

# ========== Exercise 3: Extract manufacturer (first word), convert to category, measure memory saving ==========
print("\n" + "="*60)
print("Exercise 3: Extract manufacturer (first word), convert to category, measure memory saving")
print("="*60)

# Extract first word as manufacturer
mpg['manufacturer'] = mpg['name'].str.split().str[0]
print("Unique manufacturers:", mpg['manufacturer'].unique())

# Memory before (object dtype)
mem_before = mpg['manufacturer'].memory_usage(deep=True)
print(f"Memory usage before (object): {mem_before} bytes")

# Convert to category
mpg['manufacturer_cat'] = mpg['manufacturer'].astype('category')
mem_after = mpg['manufacturer_cat'].memory_usage(deep=True)
print(f"Memory usage after (category): {mem_after} bytes")
print(f"Memory saved: {mem_before - mem_after} bytes ({100 * (mem_before - mem_after) / mem_before:.1f}% reduction)")

# ========== Exercise 4: Normalize string column (strip, lower, collapse spaces) and apply to 'name' ==========
print("\n" + "="*60)
print("Exercise 4: Normalize string column (strip, lower, collapse spaces) and apply to 'name'")
print("="*60)

def normalize_string_column(series):
    """Normalize string series: strip, lower, and collapse internal spaces."""
    return (series.str.strip()
                  .str.lower()
                  .str.replace(r'\s+', ' ', regex=True))

# Before normalization
print("Original 'name' sample (first 5):")
print(mpg['name'].head(5))

# Apply normalization
mpg['name_normalized'] = normalize_string_column(mpg['name'])
print("\nNormalized 'name' sample (first 5):")
print(mpg['name_normalized'].head(5))

# Comparison for first row
print("\nComparison for first row:")
print(f"Original: '{mpg.loc[0, 'name']}'")
print(f"Normalized: '{mpg.loc[0, 'name_normalized']}'")

# Final sample output
print("\nFinal dataset sample (relevant columns):")
print(mpg[['name', 'name_normalized', 'manufacturer', 'manufacturer_cat', 'model_date', 'horsepower']].head())

MPG dataset loaded. Shape: (398, 9)
Original data types:
mpg             float64
cylinders         int64
displacement    float64
horsepower      float64
weight            int64
acceleration    float64
model_year        int64
origin           object
name             object
dtype: object

First 5 rows:
    mpg  cylinders  displacement  horsepower  weight  acceleration  \
0  18.0          8         307.0       130.0    3504          12.0   
1  15.0          8         350.0       165.0    3693          11.5   
2  18.0          8         318.0       150.0    3436          11.0   
3  16.0          8         304.0       150.0    3433          12.0   
4  17.0          8         302.0       140.0    3449          10.5   

   model_year origin                       name  
0          70    usa  chevrolet chevelle malibu  
1          70    usa          buick skylark 320  
2          70    usa         plymouth satellite  
3          70    usa              amc rebel sst  
4          70    usa       